In [ ]:
# STEP 1: MOUNT DRIVE AND INSTALL DEPENDENCIES

from google.colab import drive
drive.mount('/content/drive')

print("Installing dependencies...")
!pip install -q google-cloud-documentai google-cloud-storage
!pip install -q google-auth google-auth-oauthlib google-auth-httplib2
!pip install -q tqdm
!pip install -q PyPDF2

print("All dependencies installed successfully!")

In [ ]:
# STEP 2: AUTHENTICATION SETUP

from google.colab import auth
import os

print("Authenticating with Google Cloud...")
auth.authenticate_user()
print("Authentication successful!")

PROJECT_ID = "Project_ID"
LOCATION = "eu"
PROCESSOR_ID = "Document_AI_Processor_ID"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

print(f"\nConfiguration:")
print(f"Project ID: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Processor ID: {PROCESSOR_ID}")

In [ ]:
# STEP 3: DEFINE HELPER FUNCTIONS

import os
import shutil
from pathlib import Path
from tqdm import tqdm
import time
from google.cloud import documentai_v1 as documentai
from google.api_core.client_options import ClientOptions
from google.api_core import exceptions
import json
from datetime import datetime

# CONFIGURATION PARAMETERS

BATCH_SIZE = 5
DELAY_BETWEEN_PDFS = 2
DELAY_BETWEEN_BATCHES = 60
MAX_RETRIES = 3
RETRY_DELAY = 30


def split_pdf_into_pages(pdf_path, max_pages=15):
    """
    Split a PDF into smaller PDFs with maximum max_pages each
    Returns list of paths to split PDFs
    """
    try:
        from PyPDF2 import PdfReader, PdfWriter
    except ImportError:
        print("Installing PyPDF2...")
        !pip install -q PyPDF2
        from PyPDF2 import PdfReader, PdfWriter

    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)

    if total_pages <= max_pages:
        return [pdf_path]

    split_pdfs = []
    base_name = os.path.splitext(os.path.basename(pdf_path))[0]
    temp_dir = os.path.join(os.path.dirname(pdf_path), '_temp_split')
    os.makedirs(temp_dir, exist_ok=True)

    chunk_num = 0
    for start_page in range(0, total_pages, max_pages):
        chunk_num += 1
        end_page = min(start_page + max_pages, total_pages)

        writer = PdfWriter()
        for page_num in range(start_page, end_page):
            writer.add_page(reader.pages[page_num])

        split_path = os.path.join(temp_dir, f"{base_name}_chunk_{chunk_num:03d}.pdf")
        with open(split_path, 'wb') as f:
            writer.write(f)

        split_pdfs.append(split_path)
        print(f"  Created chunk {chunk_num}: pages {start_page+1}-{end_page}")

    return split_pdfs


def copy_page_images_structure(source_base, target_base):
    """
    Copy page_images folders from source to target while maintaining structure
    """
    print("\nCopying page_images folders...")

    copied_count = 0
    skipped_count = 0

    for root, dirs, files in os.walk(source_base):
        if 'page_images' in dirs:
            rel_path = os.path.relpath(root, source_base)
            source_page_images = os.path.join(root, 'page_images')
            target_page_images = os.path.join(target_base, rel_path, 'page_images')

            if os.path.exists(target_page_images):
                print(f"Skipped (exists): {rel_path}")
                skipped_count += 1
                continue

            os.makedirs(os.path.dirname(target_page_images), exist_ok=True)

            try:
                shutil.copytree(source_page_images, target_page_images)
                num_files = len(os.listdir(target_page_images))
                print(f"Copied: {rel_path} ({num_files} files)")
                copied_count += 1
            except Exception as e:
                print(f"Error copying {rel_path}: {e}")

    print(f"\nSummary: Copied {copied_count} folders, Skipped {skipped_count} folders")
    return copied_count, skipped_count


def get_text_from_layout(layout, full_text):
    """
    Extract text from a layout element using text anchors
    """
    text = ""
    if hasattr(layout, 'text_anchor') and layout.text_anchor:
        for segment in layout.text_anchor.text_segments:
            start_index = int(segment.start_index) if segment.start_index else 0
            end_index = int(segment.end_index) if segment.end_index else 0
            text += full_text[start_index:end_index]
    return text


def process_pdf_with_documentai_retry(pdf_path, project_id, location, processor_id, retry_count=0):
    """
    Process a PDF with Google Document AI with exponential backoff retry logic
    Uses imageless mode for higher page limits (30 pages instead of 15)
    """
    try:
        opts = ClientOptions(api_endpoint=f"{location}-documentai.googleapis.com")
        client = documentai.DocumentProcessorServiceClient(client_options=opts)

        with open(pdf_path, "rb") as pdf_file:
            pdf_content = pdf_file.read()

        name = client.processor_path(project_id, location, processor_id)

        raw_document = documentai.RawDocument(
            content=pdf_content,
            mime_type="application/pdf"
        )

        request = documentai.ProcessRequest(
            name=name,
            raw_document=raw_document,
            imageless_mode=True
        )

        result = client.process_document(request=request)
        document = result.document

        pages_text = {}
        for page in document.pages:
            page_num = page.page_number
            page_text = ""
            if hasattr(page, 'paragraphs') and page.paragraphs:
                for paragraph in page.paragraphs:
                    paragraph_text = get_text_from_layout(paragraph.layout, document.text)
                    page_text += paragraph_text + "\n\n"
            elif hasattr(page, 'lines') and page.lines:
                for line in page.lines:
                    line_text = get_text_from_layout(line.layout, document.text)
                    page_text += line_text + "\n"
            else:
                if hasattr(page, 'blocks') and page.blocks:
                    for block in page.blocks:
                        block_text = get_text_from_layout(block.layout, document.text)
                        page_text += block_text + "\n"

            pages_text[page_num] = page_text.strip()

        return {
            'success': True,
            'pages': pages_text,
            'total_pages': len(document.pages),
            'full_text': document.text
        }

    except exceptions.ResourceExhausted as e:
        if retry_count < MAX_RETRIES:
            wait_time = RETRY_DELAY * (2 ** retry_count)
            print(f"Rate limit! Waiting {wait_time}s (retry {retry_count + 1}/{MAX_RETRIES})...")
            time.sleep(wait_time)
            return process_pdf_with_documentai_retry(pdf_path, project_id, location, processor_id, retry_count + 1)
        else:
            return {
                'success': False,
                'error': f'Rate limit exceeded after {MAX_RETRIES} retries: {str(e)}',
                'pages': {},
                'total_pages': 0
            }

    except exceptions.DeadlineExceeded as e:
        if retry_count < MAX_RETRIES:
            wait_time = RETRY_DELAY
            print(f"Timeout! Retrying in {wait_time}s ({retry_count + 1}/{MAX_RETRIES})...")
            time.sleep(wait_time)
            return process_pdf_with_documentai_retry(pdf_path, project_id, location, processor_id, retry_count + 1)
        else:
            return {
                'success': False,
                'error': f'Timeout after {MAX_RETRIES} retries: {str(e)}',
                'pages': {},
                'total_pages': 0
            }

    except Exception as e:
        return {
            'success': False,
            'error': str(e),
            'pages': {},
            'total_pages': 0
        }


def save_page_texts(pages_text, output_dir, doc_id):
    """
    Save extracted text page by page
    """
    os.makedirs(output_dir, exist_ok=True)

    for page_num, text in pages_text.items():
        page_filename = f"page_{page_num:03d}.txt"
        page_path = os.path.join(output_dir, page_filename)

        with open(page_path, 'w', encoding='utf-8') as f:
            f.write(text)

    summary = {
        'doc_id': doc_id,
        'total_pages': len(pages_text),
        'pages': [f"page_{num:03d}.txt" for num in sorted(pages_text.keys())],
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    summary_path = os.path.join(output_dir, 'extraction_info.json')
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    return True


def process_pdfs_to_text_with_batching(base_path, target_base, project_id, location, processor_id,
                                        batch_size=BATCH_SIZE,
                                        delay_between_pdfs=DELAY_BETWEEN_PDFS,
                                        delay_between_batches=DELAY_BETWEEN_BATCHES):
    """
    Process all PDFs in batches with controlled delays
    Saves extracted text to target_base directory
    """
    print("\nPDF TEXT EXTRACTION WITH DOCUMENT AI")
    print(f"Batch size: {batch_size} PDFs")
    print(f"Delay between PDFs: {delay_between_pdfs} seconds")
    print(f"Delay between batches: {delay_between_batches} seconds")
    print(f"PDF source: {base_path}")
    print(f"Text output: {target_base}")

    stats = {
        'total_found': 0,
        'processed': 0,
        'skipped': 0,
        'errors': 0,
        'total_pages': 0
    }

    pdf_files = []
    for root, dirs, files in os.walk(base_path):
        if 'pdf' in dirs:
            pdf_dir = os.path.join(root, 'pdf')
            for file in os.listdir(pdf_dir):
                if file.endswith('.pdf'):
                    pdf_path = os.path.join(pdf_dir, file)
                    rel_path = os.path.relpath(root, base_path)
                    pdf_files.append({
                        'pdf_path': pdf_path,
                        'doc_dir': root,
                        'rel_path': rel_path,
                        'doc_id': os.path.splitext(file)[0]
                    })

    stats['total_found'] = len(pdf_files)
    total_batches = (len(pdf_files) + batch_size - 1) // batch_size

    print(f"\nFound {len(pdf_files)} PDF files to process")
    print(f"Will process in {total_batches} batches\n")

    batch_num = 0
    for i in range(0, len(pdf_files), batch_size):
        batch = pdf_files[i:i + batch_size]
        batch_num += 1

        print(f"\nBATCH {batch_num}/{total_batches}")
        print(f"Processing PDFs {i+1} to {min(i+batch_size, len(pdf_files))} of {len(pdf_files)}")

        for idx, pdf_info in enumerate(batch, 1):
            try:
                from PyPDF2 import PdfReader

                reader = PdfReader(pdf_info['pdf_path'])
                pdf_pages = len(reader.pages)

                if pdf_pages > 15:
                    print(f"[{idx}/{len(batch)}] Large PDF: {pdf_info['doc_id']} ({pdf_pages} pages)")
                    print(f"             Splitting into chunks of 30 pages...")
                    split_pdfs = split_pdf_into_pages(pdf_info['pdf_path'], max_pages=30)
                else:
                    split_pdfs = [pdf_info['pdf_path']]

                rel_path = pdf_info['rel_path']
                pages_text_dir = os.path.join(target_base, rel_path, 'pages_text')

                if os.path.exists(os.path.join(pages_text_dir, 'extraction_info.json')):
                    print(f"[{idx}/{len(batch)}] Skipped: {pdf_info['doc_id']} (already done)")
                    stats['skipped'] += 1
                    continue

                all_pages_text = {}
                chunk_success = True

                for chunk_idx, chunk_pdf_path in enumerate(split_pdfs, 1):
                    chunk_label = f" (chunk {chunk_idx}/{len(split_pdfs)})" if len(split_pdfs) > 1 else ""
                    print(f"[{idx}/{len(batch)}] Processing: {pdf_info['doc_id']}{chunk_label}")

                    start_time = time.time()

                    result = process_pdf_with_documentai_retry(
                        chunk_pdf_path,
                        project_id,
                        location,
                        processor_id
                    )

                    processing_time = time.time() - start_time

                    if result['success']:
                        base_page = (chunk_idx - 1) * 30
                        for page_num, text in result['pages'].items():
                            all_pages_text[base_page + page_num] = text

                        print(f"              Chunk {chunk_idx}: {result['total_pages']} pages in {processing_time:.2f}s")
                        stats['total_pages'] += result['total_pages']

                        if chunk_idx < len(split_pdfs):
                            time.sleep(DELAY_BETWEEN_PDFS)
                    else:
                        print(f"              Chunk {chunk_idx} error: {result['error']}")
                        chunk_success = False
                        break

                if chunk_success and all_pages_text:
                    save_page_texts(
                        all_pages_text,
                        pages_text_dir,
                        pdf_info['doc_id']
                    )
                    print(f"              Saved combined text for {pdf_info['doc_id']}")
                    stats['processed'] += 1
                else:
                    stats['errors'] += 1

                if idx < len(batch):
                    time.sleep(delay_between_pdfs)

            except Exception as e:
                print(f"              Exception: {e}")
                stats['errors'] += 1

        if i + batch_size < len(pdf_files):
            print(f"\nBatch complete! Waiting {delay_between_batches}s before next batch...")
            print(f"Progress: {stats['processed']} processed, {stats['skipped']} skipped, {stats['errors']} errors")
            time.sleep(delay_between_batches)

    print("\nPROCESSING COMPLETE")
    print(f"Total PDFs found:        {stats['total_found']}")
    print(f"Successfully processed:  {stats['processed']}")
    print(f"Skipped (already done):  {stats['skipped']}")
    print(f"Errors:                  {stats['errors']}")
    print(f"Total pages extracted:   {stats['total_pages']}")

    return stats


def get_processing_progress(base_path, source_path=None):
    """
    Check which PDFs have already been processed
    """
    processed = []
    pending = []
    pdf_source = source_path if source_path else base_path

    for root, dirs, files in os.walk(pdf_source):
        if 'pdf' in dirs:
            pdf_dir = os.path.join(root, 'pdf')

            if os.path.exists(pdf_dir):
                for file in os.listdir(pdf_dir):
                    if file.lower().endswith('.pdf'):
                        doc_id = os.path.splitext(file)[0]
                        rel_path = os.path.relpath(root, pdf_source)
                        if base_path != pdf_source:
                            pages_text_dir = os.path.join(base_path, rel_path, 'pages_text')
                        else:
                            pages_text_dir = os.path.join(root, 'pages_text')

                        extraction_info = os.path.join(pages_text_dir, 'extraction_info.json')

                        if os.path.exists(extraction_info):
                            processed.append(doc_id)
                        else:
                            pending.append(doc_id)

    print("\nPROCESSING PROGRESS")
    print(f"Processed: {len(processed)} PDFs")
    print(f"Pending:   {len(pending)} PDFs")
    total = len(processed) + len(pending)
    if total > 0:
        progress_pct = (len(processed) * 100) // total
        print(f"Progress:  {len(processed)}/{total} ({progress_pct}%)")
    else:
        print(f"No PDFs found in: {pdf_source}")

    return processed, pending

print("All helper functions defined successfully!")

In [ ]:
# STEP 4: CONFIGURE YOUR PATHS

base_dir = '/content/drive/MyDrive/Final_Year/Final_Year_Project/Google_Colab'
source_path = f"{base_dir}/lk_dataset_pdf_to_image_v2/lk_legal_docs_acts"
target_path = f"{base_dir}/lk_dataset_pdf_to_pages_text_v2/lk_legal_docs_acts"

print("Paths configured:")
print(f"Source: {source_path}")
print(f"Target: {target_path}")

if os.path.exists(source_path):
    print("Source path exists")
else:
    print("Source path does NOT exist - please check the path!")

In [ ]:
# STEP 5: EXECUTE - COPY PAGE_IMAGES FOLDERS

copied, skipped = copy_page_images_structure(source_path, target_path)

print(f"\nPage images copying complete!")
print(f"Target location: {target_path}")

In [ ]:
# STEP 6: EXECUTE - PROCESS PDFs WITH DOCUMENT AI

print(f"Source path (PDFs): {source_path}")
print(f"Target path (Output): {target_path}")

print("\nChecking current progress...")
processed, pending = get_processing_progress(target_path, source_path)

print(f"\nDebug Info:")
print(f"PDF folder exists: {os.path.exists(os.path.join(source_path, 'pdf'))}")

pdf_count = 0
for root, dirs, files in os.walk(source_path):
    if 'pdf' in dirs:
        pdf_dir = os.path.join(root, 'pdf')
        pdfs = [f for f in os.listdir(pdf_dir) if f.lower().endswith('.pdf')]
        if pdfs:
            print(f"Found {len(pdfs)} PDFs in: {os.path.relpath(root, source_path)}")
            pdf_count += len(pdfs)

print(f"Total PDFs: {pdf_count}")

if len(pending) == 0:
    print("\nAll PDFs have already been processed!")
else:
    print(f"\nStarting processing of {len(pending)} pending PDFs...")

    stats = process_pdfs_to_text_with_batching(
        base_path=source_path,
        target_base=target_path,
        project_id=PROJECT_ID,
        location=LOCATION,
        processor_id=PROCESSOR_ID,
        batch_size=BATCH_SIZE,
        delay_between_pdfs=DELAY_BETWEEN_PDFS,
        delay_between_batches=DELAY_BETWEEN_BATCHES
    )

    print(f"\nAll PDF processing complete!")
    print(f"Text files saved in: {target_path}/*/pages_text/")